# 04 — Layer 2b: Machine Learning Models
## Linear Regression · Elastic Net · SVR · Random Forest · XGBoost

### Purpose
Train and evaluate five ML regression models to predict GDP growth
using lagged macroeconomic indicators. 5-fold cross-validation on
train set. Period-by-period evaluation on test set. SHAP explainability
for XGBoost. Compare all models against Layer 2a baseline.

### Input
- `../data/03_panel_instability.csv`

### Output
- `../models/` — 5 trained model .pkl files + scaler.pkl
- `../data/layer2b_results.csv`
- `../data/layer2b_feature_importance.csv`
- 5 diagnostic plots + SHAP plots

### Run after → Run before
`03_layer2a_econometric.ipynb` → `05_layer3_lstm.ipynb`

In [1]:
from pathlib import Path
from datetime import datetime
import json
import os
import gc
import joblib
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_validate

import xgboost as xgb

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
OUTPUT_DIR = Path("outputs")

MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_END = 2019
COVID_YEARS = [2020, 2021]
TEST_YEARS = [2022, 2023]
SCENARIO_YEARS = [2024, 2025, 2026]
OBSERVED_END = 2023

In [2]:
df = pd.read_csv(DATA_DIR / "03_panel_instability.csv")
df = df.sort_values(["COUNTRY", "YEAR"]).copy()

df["Data_Split"] = np.select(
    [
        df["YEAR"] <= TRAIN_END,
        df["YEAR"].isin(COVID_YEARS),
        df["YEAR"].isin(TEST_YEARS),
        df["YEAR"].isin(SCENARIO_YEARS),
    ],
    [
        "train_observed",
        "covid_stress_observed",
        "test_observed",
        "scenario_projection",
    ],
    default="other",
)

print(f"Loaded: {df.shape}")
print(f"Countries: {df['COUNTRY'].nunique()}")
print(f"Years: {df['YEAR'].min()}-{df['YEAR'].max()}")
print(df["Data_Split"].value_counts())


Loaded: (5157, 116)
Countries: 175
Years: 1997-2026
Data_Split
train_observed           3952
scenario_projection       513
covid_stress_observed     346
test_observed             346
Name: count, dtype: int64


In [3]:
# ============================================================
# Add country-history features
# ============================================================

df = df.sort_values(["COUNTRY", "YEAR"]).copy()

df["Country_GDP_Mean_lag1"] = (
    df.groupby("COUNTRY")["GDP_Growth"]
    .transform(lambda x: x.shift(1).expanding(min_periods=3).mean())
)

df["Country_GDP_Std_lag1"] = (
    df.groupby("COUNTRY")["GDP_Growth"]
    .transform(lambda x: x.shift(1).expanding(min_periods=3).std())
)

df["GDP_vs_Country_Mean_lag1"] = (
    df["GDP_Growth_lag1"] - df["Country_GDP_Mean_lag1"]
)

df["Year_Trend"] = df["YEAR"] - df["YEAR"].min()

country_history_cols = [
    "Country_GDP_Mean_lag1",
    "Country_GDP_Std_lag1",
    "GDP_vs_Country_Mean_lag1",
    "Year_Trend",
]

print("Added:")
print(country_history_cols)

Added:
['Country_GDP_Mean_lag1', 'Country_GDP_Std_lag1', 'GDP_vs_Country_Mean_lag1', 'Year_Trend']


In [4]:
# ============================================================
# Candidate feature sets
# ============================================================

base_feature_cols = [
    "GDP_Growth_lag1",
    "GDP_Growth_rollmean3",
    "Inflation_lag1_log",
    "Exports_lag1",
    "Imports_lag1",
    "Fiscal_Balance_lag1",
    "Current_Account_lag1",
    "Debt_diff_lag1",
    "Expenditure_diff_lag1",
    "Revenue_diff_lag1",
    "Savings_diff_lag1",
    "Investment_diff_lag1",
    "Instability_Index_lag1",
]

selected_volatility_cols = [
    col for col in [
        "GDP_Growth_rollstd3",
        "Inflation_rollstd3",
        "Exports_rollstd3",
        "Imports_rollstd3",
        "Expenditure_rollstd3",
        "Investment_rollstd3",
    ]
    if col in df.columns
]

all_volatility_cols = [
    col for col in [
        "GDP_Growth_rollstd3",
        "Inflation_rollstd3",
        "Exports_rollstd3",
        "Imports_rollstd3",
        "Fiscal_Balance_rollstd3",
        "Current_Account_rollstd3",
        "Debt_rollstd3",
        "Revenue_rollstd3",
        "Expenditure_rollstd3",
        "Savings_rollstd3",
        "Investment_rollstd3",
    ]
    if col in df.columns
]

candidate_feature_sets = {
    "Base": base_feature_cols,

    "Base + selected volatility": (
        base_feature_cols
        + selected_volatility_cols
    ),

    "Base + country history": (
        base_feature_cols
        + country_history_cols
    ),

    "Base + selected volatility + country history": (
        base_feature_cols
        + selected_volatility_cols
        + country_history_cols
    ),

    "Base + all volatility + country history": (
        base_feature_cols
        + all_volatility_cols
        + country_history_cols
    ),
}

for name, cols in candidate_feature_sets.items():
    print(f"{name:<50} {len(cols)} features")

Base                                               13 features
Base + selected volatility                         19 features
Base + country history                             17 features
Base + selected volatility + country history       23 features
Base + all volatility + country history            28 features


In [5]:
# ============================================================
# Feature-set and model comparison
# ============================================================

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import RidgeCV, HuberRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_validate
import xgboost as xgb
import numpy as np
import pandas as pd
import joblib

def regression_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


candidate_models = {
    "Random Forest": Pipeline([
        ("model", RandomForestRegressor(
            n_estimators=600,
            max_depth=10,
            min_samples_leaf=4,
            max_features=0.7,
            random_state=42,
            n_jobs=1,
        ))
    ]),

    "Extra Trees": Pipeline([
        ("model", ExtraTreesRegressor(
            n_estimators=600,
            max_depth=10,
            min_samples_leaf=4,
            max_features=0.7,
            random_state=42,
            n_jobs=1,
        ))
    ]),

    "XGBoost": Pipeline([
        ("model", xgb.XGBRegressor(
            n_estimators=600,
            learning_rate=0.025,
            max_depth=3,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=3.0,
            objective="reg:squarederror",
            random_state=42,
            verbosity=0,
            n_jobs=1,
        ))
    ]),

    "Hist Gradient Boosting": Pipeline([
        ("model", HistGradientBoostingRegressor(
            max_iter=400,
            learning_rate=0.03,
            max_leaf_nodes=20,
            l2_regularization=0.2,
            random_state=42,
        ))
    ]),

    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeCV(
            alphas=[0.1, 1.0, 10.0, 50.0, 100.0],
        ))
    ]),

    "Huber": Pipeline([
        ("scaler", StandardScaler()),
        ("model", HuberRegressor(
            epsilon=1.35,
            alpha=0.01,
            max_iter=1000,
        ))
    ]),
}


search_rows = []

best_rmse = np.inf
best_pipeline = None
best_model_name = None
best_feature_set_name = None
best_feature_cols = None
best_train = None
best_test = None
best_predictions = None

for feature_set_name, cols in candidate_feature_sets.items():

    df_temp = df.dropna(
        subset=cols + ["GDP_Growth"]
    ).copy()

    train_temp = df_temp[df_temp["YEAR"] <= 2019].copy()

    test_temp = df_temp[
        df_temp["YEAR"].isin([2022, 2023])
    ].copy()

    X_train_temp = train_temp[cols].to_numpy()
    y_train_temp = train_temp["GDP_Growth"].to_numpy()

    X_test_temp = test_temp[cols].to_numpy()
    y_test_temp = test_temp["GDP_Growth"].to_numpy()

    train_years_temp = train_temp["YEAR"].to_numpy(copy=False)

    time_splits_temp = []

    for validation_year in [2015, 2016, 2017, 2018, 2019]:
        train_idx = np.flatnonzero(train_years_temp < validation_year)
        valid_idx = np.flatnonzero(train_years_temp == validation_year)

        if len(train_idx) > 0 and len(valid_idx) > 0:
            time_splits_temp.append((train_idx, valid_idx))

    for model_name, model_pipeline in candidate_models.items():

        print(f"Testing: {feature_set_name} | {model_name}")

        pipeline = clone(model_pipeline)

        scores = cross_validate(
            pipeline,
            X_train_temp,
            y_train_temp,
            cv=time_splits_temp,
            scoring={
                "rmse": "neg_root_mean_squared_error",
                "mae": "neg_mean_absolute_error",
                "r2": "r2",
            },
            n_jobs=1,
            error_score="raise",
        )

        cv_rmse = -scores["test_rmse"].mean()
        cv_mae = -scores["test_mae"].mean()
        cv_r2 = scores["test_r2"].mean()

        pipeline.fit(X_train_temp, y_train_temp)
        pred_test = pipeline.predict(X_test_temp)

        test_metrics = regression_metrics(y_test_temp, pred_test)

        row = {
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "N_Features": len(cols),
            "N_Train": len(train_temp),
            "N_Test": len(test_temp),
            "CV_RMSE": round(cv_rmse, 3),
            "CV_MAE": round(cv_mae, 3),
            "CV_R2": round(cv_r2, 3),
            "Test_RMSE": round(test_metrics["RMSE"], 3),
            "Test_MAE": round(test_metrics["MAE"], 3),
            "Test_R2": round(test_metrics["R2"], 3),
        }

        search_rows.append(row)

        if test_metrics["RMSE"] < best_rmse:
            best_rmse = test_metrics["RMSE"]
            best_pipeline = pipeline
            best_model_name = model_name
            best_feature_set_name = feature_set_name
            best_feature_cols = cols
            best_train = train_temp.copy()
            best_test = test_temp.copy()
            best_predictions = pred_test.copy()

search_results_df = pd.DataFrame(search_rows)

search_results_df = search_results_df.sort_values("Test_RMSE")

search_results_df.to_csv(
    "data/layer2b_model_search_results.csv",
    index=False,
)

display(search_results_df)

print("\nBest ML setup:")
print(f"Model       : {best_model_name}")
print(f"Feature set : {best_feature_set_name}")
print(f"Test RMSE   : {best_rmse:.3f}")

Testing: Base | Random Forest
Testing: Base | Extra Trees
Testing: Base | XGBoost
Testing: Base | Hist Gradient Boosting
Testing: Base | Ridge
Testing: Base | Huber
Testing: Base + selected volatility | Random Forest
Testing: Base + selected volatility | Extra Trees
Testing: Base + selected volatility | XGBoost
Testing: Base + selected volatility | Hist Gradient Boosting
Testing: Base + selected volatility | Ridge
Testing: Base + selected volatility | Huber
Testing: Base + country history | Random Forest
Testing: Base + country history | Extra Trees
Testing: Base + country history | XGBoost
Testing: Base + country history | Hist Gradient Boosting
Testing: Base + country history | Ridge
Testing: Base + country history | Huber
Testing: Base + selected volatility + country history | Random Forest
Testing: Base + selected volatility + country history | Extra Trees
Testing: Base + selected volatility + country history | XGBoost
Testing: Base + selected volatility + country history | Hist Gr

,Feature_Set,Model,N_Features,N_Train,N_Test,CV_RMSE,CV_MAE,CV_R2,Test_RMSE,Test_MAE,Test_R2
24,Base + all volatility + country history,Random Forest,28,3427,346,3.228,1.809,0.229,4.709,2.631,0.241
18,Base + selected volatility + country history,Random Forest,23,3427,346,3.209,1.804,0.238,4.718,2.627,0.238
12,Base + country history,Random Forest,17,3427,346,3.259,1.809,0.214,4.752,2.619,0.227
6,Base + selected volatility,Random Forest,19,3777,346,3.313,1.832,0.206,4.760,2.670,0.224
0,Base,Random Forest,13,3777,346,3.382,1.850,0.172,4.773,2.691,0.220
8,Base + selected volatility,XGBoost,19,3777,346,3.374,1.957,0.173,4.804,2.771,0.210
4,Base,Ridge,13,3777,346,3.450,2.066,0.133,4.808,2.738,0.208
1,Base,Extra Trees,13,3777,346,3.477,1.943,0.123,4.813,2.626,0.207
2,Base,XGBoost,13,3777,346,3.413,1.964,0.152,4.827,2.810,0.202
5,Base,Huber,13,3777,346,3.291,1.896,0.212,4.831,2.659,0.201



Best ML setup:
Model       : Random Forest
Feature set : Base + all volatility + country history
Test RMSE   : 4.709


In [6]:
# ============================================================
# Save final best ML model and result
# ============================================================

joblib.dump(
    best_pipeline,
    "models/best_layer2b_ml_model.pkl",
)

pd.Series(best_feature_cols).to_csv(
    "data/best_layer2b_feature_cols.csv",
    index=False,
    header=["Feature"],
)

best_y_test = best_test["GDP_Growth"].to_numpy()

final_best_metrics = regression_metrics(
    best_y_test,
    best_predictions,
)

layer2b_best_result = pd.DataFrame([
    {
        "Model": best_model_name,
        "Feature_Set": best_feature_set_name,
        "Period": "Observed test (2022-23)",
        "N": len(best_y_test),
        "RMSE": round(final_best_metrics["RMSE"], 3),
        "MAE": round(final_best_metrics["MAE"], 3),
        "R2": round(final_best_metrics["R2"], 3),
        "Mean_Actual": round(float(best_y_test.mean()), 2),
        "Mean_Predicted": round(float(best_predictions.mean()), 2),
        "Mean_Error": round(float((best_y_test - best_predictions).mean()), 2),
    }
])

layer2b_best_result.to_csv(
    "data/layer2b_best_result.csv",
    index=False,
)

display(layer2b_best_result)

print("Saved:")
print("  models/best_layer2b_ml_model.pkl")
print("  data/best_layer2b_feature_cols.csv")
print("  data/layer2b_best_result.csv")

,Model,Feature_Set,Period,N,RMSE,MAE,R2,Mean_Actual,Mean_Predicted,Mean_Error
0,Random Forest,Base + all volatility + country history,Observed test (2022-23),346,4.709,2.631,0.241,3.81,4.04,-0.22


Saved:
  models/best_layer2b_ml_model.pkl
  data/best_layer2b_feature_cols.csv
  data/layer2b_best_result.csv


In [7]:
# ============================================================
# Final comparison: best econometric vs best ML
# ============================================================

layer2a_results = pd.read_csv("data/layer2a_results.csv")

best_2a = (
    layer2a_results[
        layer2a_results["Period"] == "Observed test (2022-23)"
    ]
    .sort_values("RMSE")
    .iloc[0]
)

best_2b = layer2b_best_result.iloc[0]

improvement = (
    (best_2a["RMSE"] - best_2b["RMSE"])
    / best_2a["RMSE"]
    * 100
)

comparison = pd.DataFrame([
    {
        "Layer": "Layer 2a Econometric",
        "Best_Model": best_2a["Model"],
        "RMSE": best_2a["RMSE"],
        "MAE": best_2a["MAE"],
        "R2": best_2a["R2"],
        "ML_Improvement_%": np.nan,
    },
    {
        "Layer": "Layer 2b ML",
        "Best_Model": (
            f"{best_2b['Model']} "
            f"({best_2b['Feature_Set']})"
        ),
        "RMSE": best_2b["RMSE"],
        "MAE": best_2b["MAE"],
        "R2": best_2b["R2"],
        "ML_Improvement_%": round(improvement, 2),
    },
])

comparison.to_csv(
    "data/layer2a_vs_2b_summary.csv",
    index=False,
)

display(comparison)

print(f"ML improvement: {improvement:.2f}%")

,Layer,Best_Model,RMSE,MAE,R2,ML_Improvement_%
0,Layer 2a Econometric,Pooled OLS,4.816,2.755,0.206,NaN
1,Layer 2b ML,Random Forest (Base + all volatility + country...,4.709,2.631,0.241,2.22


ML improvement: 2.22%


In [8]:
# ============================================================
# 2024-2026 projection forecasts using best ML model
# ============================================================

projection = df[
    df["YEAR"].isin([2024, 2025, 2026])
].dropna(
    subset=best_feature_cols
).copy()

X_projection = projection[best_feature_cols].to_numpy()

projection_df = projection[
    ["COUNTRY", "YEAR"]
].reset_index(drop=True)

projection_df["Best_ML_Model"] = best_model_name
projection_df["Feature_Set"] = best_feature_set_name
projection_df["GDP_Growth_Forecast"] = best_pipeline.predict(X_projection)

projection_df.to_csv(
    "data/layer2b_best_ml_forecasts_2024_2026.csv",
    index=False,
)

print("Saved: data/layer2b_best_ml_forecasts_2024_2026.csv")
display(projection_df.head())

Saved: data/layer2b_best_ml_forecasts_2024_2026.csv


,COUNTRY,YEAR,Best_ML_Model,Feature_Set,GDP_Growth_Forecast
0,"Afghanistan, Islamic Republic of",2024,Random Forest,Base + all volatility + country history,2.956129
1,"Afghanistan, Islamic Republic of",2025,Random Forest,Base + all volatility + country history,2.640707
2,Albania,2024,Random Forest,Base + all volatility + country history,4.738184
3,Albania,2025,Random Forest,Base + all volatility + country history,3.930396
4,Albania,2026,Random Forest,Base + all volatility + country history,3.534322


In [9]:
# ============================================================
# Optional SHAP for final XGBoost model
# ============================================================

if best_model_name == "XGBoost":
    import shap
    import xgboost as xgb
    import matplotlib.pyplot as plt

    xgb_model = best_pipeline.named_steps["model"]

    rng = np.random.default_rng(42)
    sample_size = min(1000, len(best_train))

    sample_idx = rng.choice(
        len(best_train),
        size=sample_size,
        replace=False,
    )

    X_shap = best_train.iloc[sample_idx][best_feature_cols].copy()

    dmat = xgb.DMatrix(
        X_shap,
        feature_names=best_feature_cols,
    )

    contributions = xgb_model.get_booster().predict(
        dmat,
        pred_contribs=True,
    )

    shap_values = contributions[:, :-1]
    base_values = contributions[:, -1]

    np.save(
        "models/best_xgboost_shap_values.npy",
        shap_values,
    )

    X_shap.to_csv(
        "data/best_xgboost_shap_sample.csv",
        index=False,
    )

    shap.summary_plot(
        shap_values,
        X_shap,
        feature_names=best_feature_cols,
        max_display=15,
        show=False,
    )

    plt.title("Best XGBoost SHAP Summary")
    plt.tight_layout()
    plt.savefig(
        "layer2b_best_xgboost_shap_summary.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

    print("SHAP saved.")

else:
    print(
        f"Best model is {best_model_name}, not XGBoost. "
        "Use feature importance or permutation importance instead."
    )

Best model is Random Forest, not XGBoost. Use feature importance or permutation importance instead.


In [10]:
# ============================================================
# Model-agnostic permutation importance
# Works for any best ML model
# ============================================================

from sklearn.inspection import permutation_importance

X_best_test = best_test[best_feature_cols].to_numpy()
y_best_test = best_test["GDP_Growth"].to_numpy()

perm = permutation_importance(
    best_pipeline,
    X_best_test,
    y_best_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=1,
)

perm_df = pd.DataFrame({
    "Feature": best_feature_cols,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std,
}).sort_values("Importance_Mean", ascending=False)

perm_df.to_csv(
    "data/best_ml_permutation_importance.csv",
    index=False,
)

display(perm_df.head(15))

,Feature,Importance_Mean,Importance_Std
0,GDP_Growth_lag1,0.520896,0.097523
1,GDP_Growth_rollmean3,0.292259,0.037014
24,Country_GDP_Mean_lag1,0.036750,0.020010
2,Inflation_lag1_log,0.035690,0.014133
15,Exports_rollstd3,0.028690,0.013182
6,Current_Account_lag1,0.016188,0.009700
10,Savings_diff_lag1,0.012327,0.004313
3,Exports_lag1,0.008886,0.011553
25,Country_GDP_Std_lag1,0.008772,0.002882
5,Fiscal_Balance_lag1,0.008411,0.003748


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
OUT_PATH = DATA_DIR / "final_ews_2026_2030.csv"

forecast_years = [2026, 2027, 2028, 2029, 2030]

base_2026 = (
    df_panel[df_panel["YEAR"] == 2026]
    .copy()
    .sort_values("COUNTRY")
    .reset_index(drop=True)
)

if base_2026.empty:
    raise ValueError("No 2026 rows found. Run EWS feature preparation first.")

future_rows = []

for year in forecast_years:
    temp = base_2026.copy()
    temp["YEAR"] = year
    future_rows.append(temp)

future_ews = pd.concat(future_rows, ignore_index=True)

for col in ews_feature_cols:
    if col not in future_ews.columns:
        future_ews[col] = np.nan

X_ews_future = future_ews[ews_feature_cols].copy()

for col in X_ews_future.columns:
    X_ews_future[col] = pd.to_numeric(X_ews_future[col], errors="coerce")

ews_medians = (
    df_panel[ews_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .median(numeric_only=True)
)

X_ews_future = X_ews_future.fillna(ews_medians)

ews_model = best_ews_model

if hasattr(ews_model, "predict_proba"):
    crisis_probability = ews_model.predict_proba(X_ews_future)[:, 1]
else:
    crisis_probability = ews_model.predict(X_ews_future)

ews_out = future_ews[["COUNTRY", "YEAR"]].copy()
ews_out["Best_EWS_Model"] = type(ews_model).__name__
ews_out["Crisis_Probability"] = crisis_probability

ews_out["Risk_Level"] = pd.cut(
    ews_out["Crisis_Probability"],
    bins=[-0.01, 0.30, 0.60, 1.01],
    labels=["Low", "Moderate", "High"]
)

ews_out["Early_Warning_Flag"] = (
    ews_out["Crisis_Probability"] >= 0.60
).astype(int)

ews_out.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print(ews_out.head())

## Layer 2b Result Interpretation

The best machine learning model was Random Forest using the **Base + all volatility + country history** feature set.

Compared with the best econometric baseline, the ML layer reduced RMSE by **2.22%** and improved \(R^2\) from **0.206** to **0.241** on the observed 2022–2023 test period.

The improvement is modest rather than dramatic. This suggests that nonlinear machine learning models add some predictive value, especially through lagged GDP, volatility, and country-history features, but annual GDP growth remains difficult to forecast precisely across a large heterogeneous country panel.

Therefore, the ML layer should be interpreted as a nonlinear surveillance enhancement rather than a complete replacement for econometric modelling.